In [359]:
import  pandas as pd
import numpy as np
import tpqoa
import matplotlib as plt
from IPython.terminal.shortcuts.auto_suggest import accept
from v20.spec_properties import account_Account



In [94]:
api = tpqoa.tpqoa('oanda.cfg')

data = api.get_history('EUR_USD', '2024-01-01 22:00:00', '2025-12-30 22:00:00', 'H1', 'A')

In [421]:
df = pd.read_csv('Dataset/eurusd1H.csv', parse_dates=['time'], index_col='time').dropna()

In [3]:
df.head()

,c,spread
time,,
2024-01-01 22:00:00,1.10438,0.00024
2024-01-01 23:00:00,1.10359,0.00017
2024-01-02 00:00:00,1.10375,0.00017
2024-01-02 01:00:00,1.10343,0.00014
2024-01-02 02:00:00,1.10181,0.00017


In [422]:
small = 50
big = 200

df['sma_small'] = df['c'].rolling(small).mean()
df['sma_big'] = df['c'].rolling(big).mean()



In [423]:
position = 0
trade_count = 0

for i in range(len(df)):
    if df['sma_small'].iloc[i] > df['sma_big'].iloc[i]:
        if position in [0, -1]:
            print(f'go long, price {df.index[i]}')
            position = 1
            trade_count += 1

    elif df['sma_small'].iloc[i] < df['sma_big'].iloc[i]:
        if position in [0, 1]:
            print(f'go short, price {df.index[i]}')
            position = -1
            trade_count +=1

print(trade_count)

go long, price 2024-01-12 05:00:00
go short, price 2024-01-16 07:00:00
go long, price 2024-02-19 02:00:00
go short, price 2024-03-01 05:00:00
go long, price 2024-03-04 21:00:00
go short, price 2024-03-15 15:00:00
go long, price 2024-04-04 12:00:00
go short, price 2024-04-11 09:00:00
go long, price 2024-04-23 13:00:00
go short, price 2024-05-01 16:00:00
go long, price 2024-05-03 04:00:00
go short, price 2024-05-23 13:00:00
go long, price 2024-05-28 07:00:00
go short, price 2024-05-30 02:00:00
go long, price 2024-06-03 11:00:00
go short, price 2024-06-10 05:00:00
go long, price 2024-06-26 02:00:00
go short, price 2024-06-26 13:00:00
go long, price 2024-07-01 08:00:00
go short, price 2024-07-22 16:00:00
go long, price 2024-08-05 04:00:00
go short, price 2024-08-29 17:00:00
go long, price 2024-09-06 11:00:00
go short, price 2024-09-10 02:00:00
go long, price 2024-09-16 01:00:00
go short, price 2024-10-01 12:00:00
go long, price 2024-10-30 15:00:00
go short, price 2024-11-06 18:00:00
go lon

In [424]:
rollup = 20
k = 2
df['mb'] = df['c'].rolling(rollup).mean()
df['ub'] = df['mb'] + (k * df['c'].rolling(rollup).std())
df['lb'] = df['mb'] - (k * df['c'].rolling(rollup).std())

In [448]:
position = 0
trade_count = 0
for i in range(len(df)):
    if df['c'].iloc[i] > df['ub'].iloc[i] and position != -1:
        print('sell')
        trade_count += 1
        position = -1
    if df['c'].iloc[i] < df['lb'].iloc[i] and position != 1:
        print('buy')
        position = 1
        trade_count += 1
print(trade_count)

buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
buy
sell
b

In [513]:
class Iterative():
    def __init__(self, start, end, amount, symbol, path, cost = True):
        self.start = start
        self.symbol = symbol
        self.trade_count = 0
        self.end = end
        self.path = path
        self.unit = 0
        self.trade = 0
        self.position = 0
        self.previous_balance = amount
        self.initial_capital = amount
        self.balance = amount
        self.get_data(path)
        self.trade_cost = cost
        self.sl = 0.001
        self.tp = 0.002
        self.entry_price = 0

    def get_data(self, path):
        datas = pd.read_csv(path, parse_dates=['time'], index_col='time').dropna()
        datas = datas.loc[self.start:self.end]
        datas['returns'] = np.log(datas['c']/datas['c'].shift(1))
        self.data = datas

    def plot_data(self, col = None):
        if col is None:
             col = 'c'
        self.data[col].plot()

    def get_values(self, bar):
        date = str(self.data.index[bar])
        price = round(self.data['c'].iloc[bar], 5)
        spread = round(self.data['spread'].iloc[bar], 5)
        return  date, price, spread

    def current_balance(self, bar):
        date, price, spread = self.get_values(bar)
        print("{} | Current Balance: {}".format(date, round(self.balance, 2)))

    def buy(self, bar, amount = None, unit = None):
        date, price, spread = self.get_values(bar)
        if self.trade_cost:
            price += spread/2
        if amount is not None:
            unit = int(amount / price)
        self.balance -= unit * price
        self.unit += unit
        self.trade += 1
        print("{} |  Buying {} for {}".format(date, unit, round(price, 5)))

    def sell(self, bar, amount = None, unit = None):
        date, price, spread = self.get_values(bar)
        if self.trade_cost:
            price -= spread/2
        if amount is not None:
            unit = int(amount / price)
        self.balance += unit * price
        self.unit -= unit
        self.trade += 1
        print("{} |  Selling {} for {}".format(date, unit, round(price, 5)))

    def print_current_position_value(self, bar):
        date, price, spread = self.get_value(bar)
        cpv = self.unit * price
        print("{} |  Current Position Value = {}".format(date, round(cpv, 2)))

    def print_current_nav(self, bar):
        date, price, spread = self.get_values(bar)
        nav = self.balance + self.unit * price
        print("{} |  Net Asset Value = {}".format(date, round(nav, 2)))

    def close_pos(self, bar):
        date, price, spread = self.get_values(bar)
        print(75 * "-")
        print("{} | +++ CLOSING FINAL POSITION +++".format(date))
        self.balance += self.unit * price # closing final position (works with short and long!)
        self.balance -= (abs(self.unit) * spread/2 * self.trade_cost)
        print("{} | closing position of {} for {}".format(date, self.unit, price))
        self.unit = 0 # setting position to neutral
        self.trade += 1
        perf = (self.balance - self.initial_capital) / self.initial_capital * 100
        print("{} | net performance (%) = {}".format(date, round(perf, 2) ))
        print("{} | number of trades executed = {}".format(date, self.trade))
        print(75 * "-")

    def show_data(self):
        return self.data


In [531]:
class IterativeBase(Iterative):

    def go_long(self, bar, unit = None, amount = None):
        if self.position == -1:
            self.buy(bar, unit = -self.unit)

            if self.balance > self.previous_balance:
                print (f'profit trade of {self.balance - self.previous_balance}')

            elif self.balance< self.previous_balance:
                print(f'loss trade of {self.balance - self.previous_balance}')

        if amount:
            if amount == "all":
                amount = self.balance
                self.previous_balance = self.balance
            self.buy(bar, amount = amount)

    # helper method
    def go_short(self, bar, unit = None, amount = None):
        if self.position == 1:
            self.sell(bar, unit = self.unit)

            if self.balance > self.previous_balance:
                print (f'profit trade of {self.balance - self.previous_balance}')
            elif self.balance< self.previous_balance:
                print(f'loss trade of {self.balance - self.previous_balance}')

        if amount:
            if amount == "all":
                amount = self.balance
            self.sell(bar, amount = amount)

    def sma(self, small = 50, big = 200):
        self.unit = 0
        self.trade = 0
        self.position = 0
        self.balance = self.initial_capital
        self.get_data(self.path)

        self.data['sma_small'] = self.data['c'].rolling(small).mean()
        self.data['sma_big'] = self.data['c'].rolling(big).mean()
        self.data = self.data.dropna()

        for i in range(len(self.data)-1):

            # adding tp and sl to buy positiions
            if self.position == 1 and self.data['c'].iloc[i] >= self.tp+self.entry_price:
                self.sell(i, unit = self.unit)
                self.position = 0
                print(f'buy tp {self.balance}')

            elif self.position == 1 and self.data['c'].iloc[i] <= self.entry_price - self.sl:
                self.buy(i, unit = self.unit)
                self.position = 0
                print(f'sl {self.balance}')

            # adding sl and tp to sell positions
            if self.position == -1 and self.data['c'].iloc[i] <= self.tp - self.entry_price:
                self.buy(i, unit = self.unit)
                self.position = 0
                print(f'sell tp {self.balance}')

            elif self.position == -1 and self.data['c'].iloc[i] >= self.entry_price + self.sl:
                self.sell(i, unit = self.unit)
                self.position = 0
                print(f'sl {self.balance}')

            if self.data['sma_small'].iloc[i] > self.data['sma_big'].iloc[i]:
                if self.position in [0, -1]:
                    self.go_long(i, amount='all')
                    self.position = 1

            elif self.data['sma_small'].iloc[i] < self.data['sma_big'].iloc[i]:
                if self.position in [0, 1]:
                    self.go_short(i, amount='all')
                    self.position = -1
        self.close_pos(-1)
        return round(self.balance, 3)

    def bollinger_band(self, rollup = 20, k = 2):
        self.get_data(self.path)
        self.position = 0
        self.trade=0
        self.data['mb'] = self.data['c'].rolling(rollup).mean()
        self.data['ub'] = self.data['mb'] + (k * self.data['c'].rolling(rollup).std())
        self.data['lb'] = self.data['mb'] - (k * self.data['c'].rolling(rollup).std())
        self.data.dropna(inplace=True)
        for i in range(len(self.data)):
            if self.data['c'].iloc[i] > self.data['ub'].iloc[i] and self.position != -1:
                print('sell')
                self.go_short(i, amount='all')
                self.trade_count += 1
                self.position = -1
            elif self.data['c'].iloc[i] < self.data['lb'].iloc[i] and self.position != 1:
                print('buy')
                self.go_long(i, amount='all')
                self.position = 1
                self.trade_count += 1

        print(self.trade_count)
        self.close_pos(-1)
        return round(self.balance, 3)


    def sma_and_bollinger_band(self, rollup = 20, k=2, small=50, big = 200):
        self.get_data(self.path)
        self.position = 0
        self.trade=0
        self.trade_count = 0
        self.balance = self.initial_capital

        self.data['mb'] = self.data['c'].rolling(rollup).mean()
        self.data['ub'] = self.data['mb'] + (k * self.data['c'].rolling(rollup).std())
        self.data['lb'] = self.data['mb'] - (k * self.data['c'].rolling(rollup).std())
        self.data['sma_small'] = self.data['c'].rolling(small).mean()
        self.data['sma_big'] = self.data['c'].rolling(big).mean()
        self.data.dropna(inplace=True)
        for i in range(len(self.data)):
            if self.data['c'].iloc[i] > self.data['ub'].iloc[i] and self.position != -1 and self.data['sma_small'].iloc[i] < self.data['sma_big'].iloc[i]:
                self.go_short(i, amount='all')
                self.trade_count += 1
                self.position = -1

            elif self.data['c'].iloc[i] < self.data['lb'].iloc[i] and self.position != 1 and self.data['sma_small'].iloc[i] > self.data['sma_big'].iloc[i]:
                self.go_long(i, amount = 'all')
                self.trade_count += 1
                self.position = 1

        print(self.trade_count)
        self.close_pos(-1)
        return round(self.balance, 3)


In [532]:
new = IterativeBase('2025', '2026', 100000, 'eurusd', 'Dataset/eurusd1H.csv')

In [533]:
new.sma()

2025-01-14 05:00:00 |  Selling 97625 for 1.02433
2025-01-14 06:00:00 |  Selling -97625 for 1.02511
sl 99923.36437500003
2025-01-14 06:00:00 |  Selling 97475 for 1.02511
2025-01-14 07:00:00 |  Selling -97475 for 1.02529
sl 99905.33150000004
2025-01-14 07:00:00 |  Selling 97440 for 1.02529
2025-01-14 08:00:00 |  Selling -97440 for 1.02599
sl 99838.09790000001
2025-01-14 08:00:00 |  Selling 97309 for 1.02599
2025-01-14 09:00:00 |  Selling -97309 for 1.02582
sl 99854.153885
2025-01-14 09:00:00 |  Selling 97340 for 1.02582
2025-01-14 10:00:00 |  Selling -97340 for 1.02585
sl 99851.233685
2025-01-14 10:00:00 |  Selling 97335 for 1.02585
2025-01-14 11:00:00 |  Selling -97335 for 1.02527
sl 99907.687985
2025-01-14 11:00:00 |  Selling 97445 for 1.02527
2025-01-14 12:00:00 |  Selling -97445 for 1.02464
sl 99968.59111
2025-01-14 12:00:00 |  Selling 97564 for 1.02464
2025-01-14 13:00:00 |  Selling -97564 for 1.02516
sl 99918.83347000001
2025-01-14 13:00:00 |  Selling 97467 for 1.02516
2025-01-14 1

np.float64(60223.781)

In [492]:
new.sma_and_bollinger_band(small=50, big = 200)

2025-01-14 15:00:00 |  Selling 97146 for 1.02937
2025-01-17 17:00:00 |  Buying 97146 for 1.0281
profit trade of 2873.0593150001077
2025-01-17 17:00:00 |  Buying 97387 for 1.0281
2025-01-30 13:00:00 |  Selling 97387 for 1.04545
profit trade of 1689.6644500000111
2025-01-30 13:00:00 |  Selling 97387 for 1.04545
2025-02-07 13:00:00 |  Buying 97387 for 1.03636
profit trade of 2573.938410000046
2025-02-07 13:00:00 |  Buying 99094 for 1.03636
2025-02-11 11:00:00 |  Selling 99094 for 1.03204
loss trade of -428.58155000001716
2025-02-11 11:00:00 |  Selling 99094 for 1.03204
2025-02-17 08:00:00 |  Buying 99094 for 1.04744
loss trade of -1954.1336800000572
2025-02-17 08:00:00 |  Buying 96181 for 1.04744
2025-03-03 11:00:00 |  Selling 96181 for 1.0451
loss trade of -224.5826350000134
2025-03-03 11:00:00 |  Selling 96181 for 1.0451
2025-03-10 08:00:00 |  Buying 96181 for 1.081
loss trade of -3677.480535000039
2025-03-10 08:00:00 |  Buying 89792 for 1.081
2025-03-25 10:00:00 |  Selling 89792 for 1.

np.float64(96966.414)